In [28]:
import h3
import pandas as pd
from pathlib import Path
import numpy as np



# Resolução do H3
RESOLUCAO = 10


In [29]:
pasta_raiz = Path(
    r"C:\Users\samuelbarroso\Downloads\Dados-Brasil\Dados"
)

dfs = []

for arquivo in pasta_raiz.rglob("*.csv"):
    try:
        #print(f"Lendo: {arquivo}")

        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            decimal=",",
            low_memory=False
        )

        dfs.append(df)

    except Exception as e:
        print(f"Erro em {arquivo}: {e}")

df_final = pd.concat(
    dfs,
    ignore_index=True
)

print(f"Total de linhas: {len(df_final):,}")

"""
caso queira, mostra os tipos em cada coluna
for col in df_final.columns:
    if df_final[col].dtype == "object":
        tipos = df_final[col].apply(type).value_counts()

        if len(tipos) > 1:
            print(f"\n{col}")
            print(tipos)"""

# Códigos e textos
df_final["br"] = df_final["br"].astype(str)

# Valores numéricos
df_final["km"] = pd.to_numeric(
    df_final["km"].astype(str).str.replace(",", "."),
    errors="coerce"
)

df_final["latitude"] = pd.to_numeric(
    df_final["latitude"].astype(str).str.replace(",", "."),
    errors="coerce"
)

df_final["longitude"] = pd.to_numeric(
    df_final["longitude"].astype(str).str.replace(",", "."),
    errors="coerce"
)

# Colunas textuais que possuem NaN
for col in [
    "tipo_acidente",
    "classificacao_acidente",
    "fase_dia",
    "condicao_metereologica",
    "regional",
    "delegacia",
    "uop"
]:
    df_final[col] = df_final[col].astype(str)


# Arredondar KM para inteiro mais próximo
df_final["km"] = (
    pd.to_numeric(
        df_final["km"].astype(str).str.replace(",", "."),
        errors="coerce"
    )
    .round()
    .astype("Int64")  # permite valores nulos
)

# Remover colunas
df_final = df_final.drop(
    columns=[
        "id",
        "dia_semana",
        "municipio",
        "delegacia",
        "uop",
        "regional",
        "ano"
    ],
    errors="ignore"  # evita erro se alguma coluna não existir
)

Total de linhas: 2,218,342


In [30]:
acidentes = df_final.copy()

In [31]:
# Base de kms das rodovias
rodovias = pd.read_parquet(
    r"C:\Users\samuelbarroso\Downloads\202604A\rodovias_km.parquet"
)

# Garantir tipos compatíveis
acidentes["br"] = acidentes["br"].astype(str)
rodovias["br"] = rodovias["br"].astype(str)

acidentes["km"] = pd.to_numeric(acidentes["km"], errors="coerce").astype("Int64")
rodovias["km"] = pd.to_numeric(rodovias["km"], errors="coerce").astype("Int64")

# Merge
acidentes = acidentes.merge(
    rodovias,
    on=["uf", "br", "km"],
    how="left",
    suffixes=("", "_rodovia")
)

# Preencher apenas onde faltava coordenada
acidentes["latitude"] = acidentes["latitude"].fillna(
    acidentes["latitude_rodovia"]
)

acidentes["longitude"] = acidentes["longitude"].fillna(
    acidentes["longitude_rodovia"]
)

# Remover colunas auxiliares
acidentes = acidentes.drop(
    columns=[
        "latitude_rodovia",
        "longitude_rodovia"
    ]
)

print(
    "Latitude faltando:",
    acidentes["latitude"].isna().sum()
)

print(
    "Longitude faltando:",
    acidentes["longitude"].isna().sum()
)
antes = len(acidentes)

acidentes = acidentes.dropna(
    subset=["latitude", "longitude"]
)

depois = len(acidentes)

print(f"Removidos: {antes - depois:,}")
print(f"Restantes: {depois:,}")


acidentes["grau_severidade"] = np.select(
    [
        acidentes["mortos"] > 0,
        acidentes["feridos_graves"] > 0,
        acidentes["feridos_leves"] > 0
    ],
    [
        4,
        3,
        2
    ],
    default=1
)

# Remover colunas
acidentes = acidentes.drop(
    columns=[
        "uf",
        "br",
        "km",
        "causa_acidente",
        "tipo_acidente",
        "classificacao_acidente",
        "veiculos",
        "pessoas",
        "mortos",
        "feridos_leves",
        "feridos graves",
        "ilesos",
        "ignorados",
        "feridos",
        "feridos_graves"
    ],
    errors="ignore"  # evita erro se alguma coluna não existir
)

Latitude faltando: 9593
Longitude faltando: 9593
Removidos: 9,593
Restantes: 2,208,749


In [32]:
acidentes.to_csv(
    r"C:\Users\samuelbarroso\Downloads\dados_unificados2.csv",
    index=False
)
acidentes.to_parquet(
    r"C:\Users\samuelbarroso\Downloads\dados_unificados2.parquet",
    index=False
)

In [33]:
# Remover colunas
acidentes = acidentes.drop(
    columns=[
        "fase dia",
        "sentido via",
        "condicao_metereologica",
        "tipo pista",
        "tracado_via",
        "uso_solo",
        "tipo_pista",
        "sentido_via",
        "fase_dia"
    ],
    errors="ignore"  # evita erro se alguma coluna não existir
)
acidentes["pais"] = "BR"

In [35]:
acidentes.to_csv(
    r"C:\Users\samuelbarroso\Downloads\dados_unificados3.csv",
    index=False
)
acidentes.to_parquet(
    r"C:\Users\samuelbarroso\Downloads\dados_unificados3.parquet",
    index=False
)

In [36]:
usecols = [
        'Severity', 'Start_Time', 'Start_Lat', 'Start_Lng'    ]
# Acidentes
acidentes = pd.read_parquet(
    r"C:\Users\samuelbarroso\Documents\Desenvolvimento\TraficGenius\dataset\dataset_consolidado.parquet"
, columns=usecols)
acidentes["pais"] = "US"



In [37]:

COLUMNS_MAPPING = {
    'Severity': 'grau_severidade',
    'Start_Time': 'data_inversa',
    'Start_Lat': 'latitude',
    'Start_Lng': 'longitude'
}
acidentes = acidentes.rename(columns=COLUMNS_MAPPING)
acidentes["data_inversa"] = pd.to_datetime(
    acidentes["data_inversa"],
    format="mixed"
)

acidentes["data"] = acidentes["data_inversa"].dt.date
acidentes["hora"] = acidentes["data_inversa"].dt.time
# Remover colunas
acidentes = acidentes.drop(
    columns=[
        "data_inversa"
    ],
    errors="ignore"  # evita erro se alguma coluna não existir
)
COLUMNS_MAPPING = {
    'data': 'data_inversa',
    'hora': 'horario'
}
acidentes = acidentes.rename(columns=COLUMNS_MAPPING)

In [39]:
acidentes.to_csv(
    r"C:\Users\samuelbarroso\Downloads\dados_unificados4.csv",
    index=False
)
acidentes.to_parquet(
    r"C:\Users\samuelbarroso\Downloads\dados_unificados4.parquet",
    index=False
)

In [ ]:
df1 = pd.read_parquet(r"C:\Users\samuelbarroso\Downloads\dados_unificados4.parquet")
df2 = pd.read_parquet(r"C:\Users\samuelbarroso\Downloads\dados_unificados3.parquet")

acidentes = pd.concat([df1, df2], ignore_index=True)

antes = len(acidentes)

acidentes = acidentes.drop_duplicates()

depois = len(acidentes)
print(f"Removidos: {antes - depois:,}")
print(f"Restantes: {depois:,}")

(9937143, 6)


In [47]:
acidentes["horario"] = pd.to_datetime(
    acidentes["horario"].astype(str),
    format="%H:%M:%S"
)

acidentes["horario_r"] = acidentes["horario"].dt.round("h").dt.time

In [49]:
acidentes

,grau_severidade,latitude,longitude,pais,data_inversa,horario,horario_r
0,3,39.865147,-84.058723,US,2016-02-08,1900-01-01 05:46:00,06:00:00
1,2,39.928059,-82.831184,US,2016-02-08,1900-01-01 06:07:59,06:00:00
2,2,39.063148,-84.032608,US,2016-02-08,1900-01-01 06:49:27,07:00:00
3,3,39.747753,-84.205582,US,2016-02-08,1900-01-01 07:23:34,07:00:00
4,2,39.627781,-84.188354,US,2016-02-08,1900-01-01 07:39:07,08:00:00
...,...,...,...,...,...,...,...
9937138,2,-25.430412,-49.347344,BR,2026-02-27,1900-01-01 22:30:00,22:00:00
9937139,3,-23.031173,-44.130535,BR,2026-02-28,1900-01-01 09:40:00,10:00:00
9937140,3,-14.266993,-39.326577,BR,2026-04-17,1900-01-01 18:00:00,18:00:00
9937141,2,-20.007164,-44.215484,BR,2026-04-16,1900-01-01 08:15:00,08:00:00


In [48]:
acidentes.dtypes

grau_severidade             int64
latitude                  float64
longitude                 float64
pais                       object
data_inversa               object
horario            datetime64[ns]
horario_r                  object
dtype: object

In [ ]:

# Criar coluna H3
acidentes["h3"] = acidentes.apply(
    lambda x: h3.latlng_to_cell(
        x["latitude"],
        x["longitude"],
        RESOLUCAO
    ),
    axis=1
)

# Hexágonos únicos
hex_unicos = pd.DataFrame({
    "h3": acidentes["h3"].unique()
})

# Centro de cada hexágono
centros = [h3.cell_to_latlng(h) for h in hex_unicos["h3"]]

hex_unicos["lat_centro"] = [c[0] for c in centros]
hex_unicos["lon_centro"] = [c[1] for c in centros]

# Juntar de volta
acidentes = acidentes.merge(
    hex_unicos,
    on="h3",
    how="left"
)

Removidos: 0
Restantes: 7,240,402


In [12]:
len(hex_unicos)

1334906

In [148]:
acidentes

,grau_severidade,latitude,longitude,data_inversa,horario,h3,lat_centro,lon_centro
0,3,39.865147,-84.058723,2016-02-08,05:46:00,8b2a90741940fff,39.865048,-84.058804
1,2,39.928059,-82.831184,2016-02-08,06:07:59,8b2a9565691dfff,39.927954,-82.831077
2,2,39.063148,-84.032608,2016-02-08,06:49:27,8b2a93aab2b5fff,39.063283,-84.032517
3,3,39.747753,-84.205582,2016-02-08,07:23:34,8b2a906550e3fff,39.747801,-84.205667
4,2,39.627781,-84.188354,2016-02-08,07:39:07,8b2a939a3c40fff,39.627877,-84.188520
...,...,...,...,...,...,...,...,...
7240397,2,34.002480,-117.379360,2019-08-23,18:03:25,8b29a0070ac1fff,34.002381,-117.379493
7240398,2,32.766960,-117.148060,2019-08-23,19:11:30,8b29a41ab251fff,32.766811,-117.148103
7240399,2,33.775450,-117.847790,2019-08-23,19:00:21,8b29a0b541b1fff,33.775496,-117.847748
7240400,2,33.992460,-118.403020,2019-08-23,19:00:21,8b29a19b3a43fff,33.992501,-118.402727


In [74]:
# Base de kms das rodovias
acidentes.to_parquet(
    r"C:\Users\samuelbarroso\Downloads\202604A\teste.parquet"
)
